In [3]:
import re
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

FILE = "../Data/PS_ Net Debt (excluding public sector banks) as a % of GDP_ NSA-170926.csv"

# The file stacks metadata + annual + quarterly + monthly rows.
raw = pd.read_csv(FILE, header=None, names=["label", "value"], dtype=str)
raw["value"] = pd.to_numeric(raw["value"], errors="coerce")

# Keep only MONTHLY rows: label like "1993 MAR", "2020 APR" (4-digit year + 3-letter month).
monthly = raw[raw["label"].str.match(r"^\d{4}\s[A-Z]{3}$", na=False)].copy()
monthly["date"] = pd.to_datetime(monthly["label"], format="%Y %b", errors="coerce")
df = monthly.dropna(subset=["date", "value"]).sort_values("date")
df["series"] = "debt"

events = pd.DataFrame({
    "date": pd.to_datetime(["2008-09", "2020-03"], format="%Y-%m"),
    "label": ["Financial crisis", "..COVID-19"],
})

ev_rule = alt.Chart(events).mark_rule(
    color="#94a3b8", strokeDash=[2, 3], strokeWidth=1
).encode(x="date:T")

ev_text = alt.Chart(events).mark_text(
    align="left", dx=4, dy=-4, fontSize=10, color="#64748b"
).encode(x="date:T", y=alt.value(10), text="label:N")

line = alt.Chart(df).mark_line(strokeWidth=1.4).encode(
    x=alt.X("date:T", axis=alt.Axis(format="%Y", tickCount=8), title=None),
    y=alt.Y("value:Q", scale=alt.Scale(domain=[0, 100]), title="Net debt (% of GDP)"),
    color=alt.Color("series:N", legend=None),
)

caption = alt.Title(
    text="Source: ONS",
    subtitle=[
        "Public sector net debt excluding public sector banks, % of GDP. Monthly, 1993–2026.",
        "From ~36% before the financial crisis to ~94% today.",
    ],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12,
)

chart = (
    (ev_rule + ev_text + line)
    .properties(
        width=700,
        height=270,
        title=caption,
        # >>> the fix: force the OUTER svg to 700x270 (view shrinks to fit axes + caption)
        autosize=alt.AutoSizeParams(type="fit", contains="padding"),
    )
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor="#676A86", titleColor="#676A86")
)

# Save via the in-house wrapper
styles.save(chart, path=".", name="britain_debt", svg=True)

# --- Diagnostic: confirm the exported box is really 700x270 ---

chart.save("britain_debt_direct.png", scale_factor=2)
from PIL import Image
print("direct PNG ->", Image.open("britain_debt_direct.png").size)   # expect (1400, 540)

def root_dims(path):
    head = open(path, encoding="utf-8").read(400)
    w = re.search(r'\bwidth="([\d.]+)"', head)
    h = re.search(r'\bheight="([\d.]+)"', head)
    return (w.group(1) if w else "?", h.group(1) if h else "?")

print("direct  chart.save()  ->", root_dims("britain_debt_direct.svg"))
try:
    print("ecostyles styles.save ->", root_dims("britain_debt.svg"))
except FileNotFoundError:
    print("ecostyles styles.save -> (check the exact filename it wrote)")

chart

direct PNG -> (1400, 540)
direct  chart.save()  -> ('700', '270')
ecostyles styles.save -> ('350', '280')


alt.LayerChart(...)